# 138 · Vision QA Agent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/main/examples/138-vision-qa-agent/vision_qa_agent_workbook.ipynb)

**What this notebook teaches:** How to send images to a multimodal LLM using base64 encoding, how to build the OpenAI vision message payload, and how a LangGraph single-node workflow ties the whole pipeline together.

**Two halves:**
- **Part A (cells 1–4):** Pure Python — run anywhere, no API key needed. You’ll see base64 encoding, URL fetching, and message payload construction without touching the API.
- **Part B (cells 5–7):** Requires `OPENAI_API_KEY`. Runs three questions against a public image through the full LangGraph workflow.

| Concept | Why it matters |
|---------|----------------|
| Base64 image encoding | The OpenAI API accepts images as `data:<mime>;base64,<string>` inline URIs |
| URL vs file path handling | `load_image_b64` branches on `source.startswith("http")` |
| Multimodal `content` list | A vision message mixes `image_url` and `text` blocks in one `content` array |
| MIME type detection | JPEG, PNG, and WebP each need the correct MIME header |
| Fail-fast setup | Part B checks connectivity before running the workflow |


In [ ]:
%pip install -q openai httpx python-dotenv langgraph


## Part A · Cell 1 — Base64 Encoding: Why and How

HTTP APIs are text protocols. Images are binary. **Base64** converts arbitrary binary data into a safe ASCII string by encoding every 3 bytes as 4 printable characters (~33% size overhead).

When you send an image to GPT-4o-mini you have two options:
1. **Public URL** — pass `{"url": "https://..."}` and let OpenAI fetch it.
2. **Inline base64** — pass `{"url": "data:image/jpeg;base64,<string>"}` so the image travels inside the request body.

Option 2 is necessary for private images, local files, or when you can’t guarantee the URL is publicly reachable.

```
Binary bytes:  \xff\xd8\xff\xe0\x00\x10
Base64 string: /9j/4AAQ...  (safe ASCII)
Data URI:      data:image/jpeg;base64,/9j/4AAQ...
```

**MIME type** tells the model how to decode the bytes. The most common types:

| Extension | MIME type |
|-----------|----------|
| `.jpg` / `.jpeg` | `image/jpeg` |
| `.png` | `image/png` |
| `.webp` | `image/webp` |
| `.gif` | `image/gif` |


In [ ]:
# Part A · Cell 2 — Fetch an image from a URL and show its base64 encoding
# No API key needed — pure Python + httpx

import base64
import httpx

# A small public PNG (Wikipedia transparency demo, ~12 KB)
IMAGE_URL = (
    "https://upload.wikimedia.org/wikipedia/commons/thumb/4/47/"
    "PNG_transparency_demonstration_1.png/"
    "240px-PNG_transparency_demonstration_1.png"
)

resp = httpx.get(IMAGE_URL, timeout=10)
resp.raise_for_status()

# Detect MIME type from response headers
mime = resp.headers.get("content-type", "image/png").split(";")[0]
raw_bytes = resp.content

# Base64 encode
b64_string = base64.b64encode(raw_bytes).decode()

print(f"Image URL    : {IMAGE_URL}")
print(f"Content-Type : {mime}")
print(f"Raw size     : {len(raw_bytes):,} bytes")
print(f"Base64 size  : {len(b64_string):,} chars  (+{(len(b64_string)/len(raw_bytes)-1)*100:.0f}% overhead)")
print(f"First 60 chars of b64: {b64_string[:60]}...")

# Build the data URI that the API expects
data_uri = f"data:{mime};base64,{b64_string}"
print(f"\nData URI prefix: {data_uri[:50]}...")


## Part A · Cell 3 — The Vision Message Structure

A standard chat message has `{"role": "user", "content": "some text"}`. For vision, `content` becomes a **list of typed blocks**:

```python
{
    "role": "user",
    "content": [
        {
            "type": "image_url",
            "image_url": {
                "url": "data:image/png;base64,iVBOR...",
                # optional: "detail": "low" | "high" | "auto"
            }
        },
        {
            "type": "text",
            "text": "What do you see in this image?"
        }
    ]
}
```

Key rules:
- **Order matters** — image block first, text question second. The model reads the image before interpreting your question.
- **`detail` parameter** — `"low"` uses a fixed 85-token budget (fast, good for thumbnails); `"high"` tiles the image into 512px crops (accurate, expensive for large images). Default is `"auto"`.
- **Multiple images** are supported — add more `image_url` blocks to the list.


In [ ]:
# Part A · Cell 4 — Build the vision message payload (no API call)
# This is exactly what src/tools.py does in vision_content()

import json
from pathlib import Path

MIME_MAP = {
    "jpg": "image/jpeg",
    "jpeg": "image/jpeg",
    "png": "image/png",
    "webp": "image/webp",
}


def load_image_b64(source: str) -> tuple:
    """Return (b64_string, mime_type) from a URL or local file path."""
    if source.startswith("http"):
        resp = httpx.get(source, timeout=10)
        mime = resp.headers.get("content-type", "image/jpeg").split(";")[0]
        return base64.b64encode(resp.content).decode(), mime
    path = Path(source)
    ext = path.suffix.lstrip(".").lower()
    mime = MIME_MAP.get(ext, "image/jpeg")
    return base64.b64encode(path.read_bytes()).decode(), mime


def vision_content(source: str, text: str) -> list:
    """Build the multimodal content list for the OpenAI messages API."""
    b64, mime = load_image_b64(source)
    return [
        {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{b64}"}},
        {"type": "text", "text": text},
    ]


# Build the payload using the image we already fetched
content = vision_content(IMAGE_URL, "What colors are most prominent?")

# Inspect the structure without printing the full base64 blob
display_content = [
    {
        "type": content[0]["type"],
        "image_url": {"url": content[0]["image_url"]["url"][:60] + "...  [truncated]"},
    },
    content[1],
]

full_message = {"role": "user", "content": display_content}
print("Full vision message structure:")
print(json.dumps(full_message, indent=2))

# Show how many tokens the image block occupies in the payload
b64, mime = load_image_b64(IMAGE_URL)
print(f"\nImage block size in request: ~{len(b64):,} characters of base64")
print(f"Estimated base64 size in KB: {len(b64)/1024:.1f} KB")


---
## Part B · Live API Run

The cells below require an active `OPENAI_API_KEY` in your `.env` file (or set as an environment variable).

```bash
# .env
OPENAI_API_KEY=sk-...
```

Cell 5 checks connectivity **fail-fast** — it will print a clear error and raise if the key is missing or the API is unreachable before you run the expensive cells.


In [ ]:
# Part B · Cell 5 — Fail-fast setup (checks API key + connectivity)
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.environ.get("OPENAI_API_KEY", "")
if not api_key:
    raise EnvironmentError(
        "OPENAI_API_KEY not set.\n"
        "Add it to your .env file or set the environment variable, then re-run this cell."
    )

client = OpenAI(api_key=api_key)

# Quick connectivity probe — list models is a cheap, read-only call
try:
    models = client.models.list()
    vision_models = [m.id for m in models if "gpt-4o" in m.id]
    print(f"Connected to OpenAI API.")
    print(f"Available vision-capable models: {vision_models[:4]}")
except Exception as e:
    raise ConnectionError(f"Cannot reach OpenAI API: {e}") from e


In [ ]:
# Part B · Cell 6 — Run the LangGraph workflow against 3 questions
import sys
sys.path.insert(0, "../..")

# Inline the workflow so this cell is self-contained
from langgraph.graph import StateGraph, END
from typing import TypedDict


class VisionState(TypedDict):
    image_source: str
    question: str
    answer: str


def answer_node(state: VisionState, config: dict) -> VisionState:
    _client = config["configurable"]["client"]
    model = config["configurable"].get("model", "gpt-4o-mini")
    b64, mime = load_image_b64(state["image_source"])
    content = [
        {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{b64}"}},
        {"type": "text", "text": state["question"]},
    ]
    resp = _client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": content}],
        max_tokens=512,
    )
    return {**state, "answer": resp.choices[0].message.content}


def build_workflow():
    graph = StateGraph(VisionState)
    graph.add_node("answer", answer_node)
    graph.set_entry_point("answer")
    graph.add_edge("answer", END)
    return graph.compile()


workflow = build_workflow()
cfg = {"configurable": {"client": client, "model": "gpt-4o-mini"}}

QUESTIONS = [
    "What do you see in this image?",
    "What colors are most prominent?",
    "Describe the geometric shapes present.",
]

print(f"Image: {IMAGE_URL}\n")
print("=" * 60)
for q in QUESTIONS:
    result = workflow.invoke(
        {"image_source": IMAGE_URL, "question": q, "answer": ""},
        config=cfg,
    )
    print(f"Q: {q}")
    print(f"A: {result['answer']}")
    print("-" * 60)


In [ ]:
# Part B · Cell 7 — Try your own image and question
# Swap in any public image URL or a local file path

MY_IMAGE = IMAGE_URL  # replace with your own URL or "/path/to/image.jpg"
MY_QUESTION = "Is there any text visible in this image? If so, what does it say?"

result = workflow.invoke(
    {"image_source": MY_IMAGE, "question": MY_QUESTION, "answer": ""},
    config=cfg,
)
print(f"Q: {MY_QUESTION}")
print(f"A: {result['answer']}")
